MVP | Data Engineering\
Pipeline de Dados: Veículos Elétricos vs. Combustão\
Fatores Associados à Adoção de EV no Mundo (2010-2025)\
Aluno: João Guilherme Cartaxo Loureiro\
Matricula: 4052026000083

Coleta e Carregamento de Dados

In [0]:
#Leitura do arquivo bruto com Spark e conferência da forma
df_raw = spark.table("default.ev_vs_petrol_dataset_v_3")
 
print("Linhas:", df_raw.count(), "| Colunas:", len(df_raw.columns))
df_raw.printSchema()
display(df_raw.limit(5))

Modelagem de Dados

In [0]:
#Confirmação do grão país × ano × segmento
from pyspark.sql import functions as F
 
total = df_raw.count()
distintos = df_raw.select("country", "year", "vehicle_segment").distinct().count()
print("Linhas:", total, "| Combinações distintas:", distintos)  # 1200 e 1200 -> grão confirmado

In [0]:
#Teste de variação das colunas de contexto por segmento
macro = ["gdp_per_capita","fuel_price_usd_per_liter","electricity_price_usd_per_kwh",
         "urban_population_percent","co2_emissions_transport_mt","emission_regulation_score",
         "ev_subsidy_usd","charging_stations","avg_ev_range_km","fast_chargers_share"]
 
for c in macro:
    n = (df_raw.groupBy("country","year").agg(F.countDistinct(c).alias("n"))
              .filter("n > 1").count())
    print(f"{c:32s} -> país-ano variando por segmento: {n}")  # esperado: 0 para todas

Catálogo de Dados

In [0]:
#Geração dos domínios do catálogo por código
num_cols = ["year","ev_sales","petrol_car_sales","diesel_car_sales","total_vehicle_sales",
            "ev_market_share","charging_stations","fast_chargers_share","avg_ev_range_km",
            "fuel_price_usd_per_liter","electricity_price_usd_per_kwh","gdp_per_capita",
            "urban_population_percent","co2_emissions_transport_mt","ev_subsidy_usd",
            "emission_regulation_score","ev_growth_rate_yoy"]
cat_cols = ["country","region","vehicle_segment","powertrain_type","is_ev_dominant"]
 
display(df_raw.select([F.min(c).alias(f"min_{c}") for c in num_cols] +
                      [F.max(c).alias(f"max_{c}") for c in num_cols]))
for c in cat_cols:
    print(c, "=>", [r[0] for r in df_raw.select(c).distinct().collect()])

Criação do Schema e Camada Bronze

In [0]:
#Criação do schema e persistência da camada bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS ev_mvp")
spark.sql("USE ev_mvp")

In [0]:
#Bronze: dados crus, sem alteração
(df_raw.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable("ev_mvp.bronze_vendas"))

Camada Silver - Limpeza e Conformação

In [0]:
#Transformações da camada silver
df_silver = (spark.table("ev_mvp.bronze_vendas")
    .withColumn("country",         F.trim("country"))
    .withColumn("region",          F.trim("region"))
    .withColumn("vehicle_segment", F.lower(F.trim("vehicle_segment")))
    .withColumn("year",            F.col("year").cast("int"))
    .withColumn("ev_sales",        F.col("ev_sales").cast("long"))
    .withColumn("total_vehicle_sales", F.col("total_vehicle_sales").cast("long")))
 
(df_silver.write.format("delta").mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable("ev_mvp.silver_vendas"))


Camada Gold - Dimensões com Surrogate Keys

In [0]:
#Construção das dimensões com chaves substitutas
from pyspark.sql.window import Window
 
df_silver = spark.table("ev_mvp.silver_vendas")
 
dim_tempo = (df_silver.select("year").distinct()
    .withColumn("sk_tempo", F.row_number().over(Window.orderBy("year")))
    .withColumn("decada", (F.floor(F.col("year")/10)*10).cast("int"))
    .withColumn("periodo", F.when(F.col("year") >= 2020, "pos_2020").otherwise("pre_2020")))
 
dim_geografia = (df_silver.select("country","region").distinct()
    .withColumn("sk_geografia", F.row_number().over(Window.orderBy("region","country"))))
 
dim_segmento = (df_silver.select("vehicle_segment").distinct()
    .withColumn("sk_segmento", F.row_number().over(Window.orderBy("vehicle_segment"))))
 
for nome, dim in [("dim_tempo", dim_tempo), ("dim_geografia", dim_geografia),
                  ("dim_segmento", dim_segmento)]:
    (dim.write.format("delta").mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(f"ev_mvp.{nome}"))


Camada Gold - Fato de Vendas (país × ano × segmento)

In [0]:
#Construção do fato de vendas
fato_vendas = (df_silver
    .join(dim_tempo.select("year","sk_tempo"), "year")
    .join(dim_geografia.select("country","region","sk_geografia"), ["country","region"])
    .join(dim_segmento.select("vehicle_segment","sk_segmento"), "vehicle_segment")
    .select("sk_tempo","sk_geografia","sk_segmento",
            "ev_sales","petrol_car_sales","diesel_car_sales","total_vehicle_sales",
            "ev_market_share","ev_growth_rate_yoy","is_ev_dominant"))
 
(fato_vendas.write.format("delta").mode("overwrite").option("overwriteSchema","true")
            .saveAsTable("ev_mvp.fato_vendas"))
print("fato_vendas:", spark.table("ev_mvp.fato_vendas").count(), "linhas (esperado 1200)")


Camada gold - Fato de Contexto (país × ano)

In [0]:
#Construção do fato de contexto por país-ano
macro_cols = ["gdp_per_capita","fuel_price_usd_per_liter","electricity_price_usd_per_kwh",
              "urban_population_percent","co2_emissions_transport_mt","emission_regulation_score",
              "ev_subsidy_usd","charging_stations","avg_ev_range_km","fast_chargers_share"]
 
fato_contexto = (df_silver
    .select(["country","region","year"] + macro_cols)
    .dropDuplicates(["country","year"])
    .join(dim_tempo.select("year","sk_tempo"), "year")
    .join(dim_geografia.select("country","region","sk_geografia"), ["country","region"])
    .select(["sk_tempo","sk_geografia"] + macro_cols))
 
(fato_contexto.write.format("delta").mode("overwrite").option("overwriteSchema","true")
              .saveAsTable("ev_mvp.fato_contexto_pais_ano"))
print("contexto:", spark.table("ev_mvp.fato_contexto_pais_ano").count(), "linhas (esperado 400)")


Análise de Dados

In [0]:
#Teste de nulos e duplicatas
b = spark.table("ev_mvp.bronze_vendas")
display(b.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in b.columns]))  # esperado 0
print("Duplicatas completas:", b.count() - b.dropDuplicates().count())                # esperado 0


In [0]:
#Teste de integridade (vendas de veículos)
inconsistentes = b.filter(
    (F.col("ev_sales") + F.col("petrol_car_sales") + F.col("diesel_car_sales"))
    != F.col("total_vehicle_sales")).count()
print("Linhas onde ev+petrol+diesel != total:", inconsistentes)  # esperado 0


In [0]:
#Teste de colunas redundantes
chk = b.withColumn("share_calc", F.round(F.col("ev_sales")/F.col("total_vehicle_sales")*100, 2))
print("Divergências de market_share:",
      chk.filter(F.abs(F.col("share_calc")-F.col("ev_market_share")) > 0.01).count())
display(b.groupBy("is_ev_dominant").agg(F.min("ev_market_share"), F.max("ev_market_share")))

In [0]:
#Verificação de outliers
display(b.select(F.min("ev_growth_rate_yoy"), F.max("ev_growth_rate_yoy")))
display(b.filter("ev_growth_rate_yoy >= 300").select("country","year","ev_sales","ev_growth_rate_yoy"))


Solução do Problema Proposto

Q1 - Participação de EV por Região e Período

In [0]:
%sql
SELECT g.region, t.periodo, ROUND(AVG(f.ev_market_share), 2) AS share_medio
FROM ev_mvp.fato_vendas f
JOIN ev_mvp.dim_geografia g ON f.sk_geografia = g.sk_geografia
JOIN ev_mvp.dim_tempo     t ON f.sk_tempo     = t.sk_tempo
GROUP BY g.region, t.periodo
ORDER BY g.region, t.periodo

Q2 - Ano em que cada país se tornou EV-dominante

In [0]:
%sql
SELECT g.country, MIN(t.year) AS ano_dominante
FROM ev_mvp.fato_vendas f
JOIN ev_mvp.dim_geografia g ON f.sk_geografia = g.sk_geografia
JOIN ev_mvp.dim_tempo     t ON f.sk_tempo     = t.sk_tempo
WHERE f.is_ev_dominant = 1
GROUP BY g.country
ORDER BY ano_dominante;


Q3 - Adoção vs. subsídio e infraestrutura de recarga

In [0]:
%sql
SELECT g.country, t.year,
       AVG(v.ev_market_share) AS share_ev,
       c.ev_subsidy_usd, c.charging_stations
FROM ev_mvp.fato_vendas v
JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
JOIN ev_mvp.fato_contexto_pais_ano c
     ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
GROUP BY g.country, t.year, c.ev_subsidy_usd, c.charging_stations
ORDER BY share_ev DESC
LIMIT 20;

In [0]:
%sql
SELECT
  ROUND(corr(share_ev, ev_subsidy_usd),    3) AS corr_subsidio,
  ROUND(corr(share_ev, charging_stations), 3) AS corr_recarga
FROM (
  SELECT g.country, t.year,
         AVG(v.ev_market_share) AS share_ev,
         c.ev_subsidy_usd,
         c.charging_stations
  FROM ev_mvp.fato_vendas v
  JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
  JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
  JOIN ev_mvp.fato_contexto_pais_ano c
       ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
  GROUP BY g.country, t.year, c.ev_subsidy_usd, c.charging_stations
);

Q4 - Preço relativo combustível × eletricidade vs. adoção

In [0]:
%sql
SELECT g.country, t.year,
       AVG(v.ev_market_share) AS share_ev,
       c.fuel_price_usd_per_liter, c.electricity_price_usd_per_kwh,
       ROUND(c.fuel_price_usd_per_liter / c.electricity_price_usd_per_kwh, 2) AS razao_fuel_elec
FROM ev_mvp.fato_vendas v
JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
JOIN ev_mvp.fato_contexto_pais_ano c
     ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
GROUP BY g.country, t.year, c.fuel_price_usd_per_liter, c.electricity_price_usd_per_kwh
ORDER BY share_ev DESC;

In [0]:
%sql
SELECT
  ROUND(corr(share_ev, fuel_price_usd_per_liter),      3) AS corr_combustivel,
  ROUND(corr(share_ev, electricity_price_usd_per_kwh), 3) AS corr_eletricidade,
  ROUND(corr(share_ev, razao_fuel_elec),               3) AS corr_razao
FROM (
  SELECT g.country, t.year,
         AVG(v.ev_market_share) AS share_ev,
         c.fuel_price_usd_per_liter,
         c.electricity_price_usd_per_kwh,
         c.fuel_price_usd_per_liter / c.electricity_price_usd_per_kwh AS razao_fuel_elec
  FROM ev_mvp.fato_vendas v
  JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
  JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
  JOIN ev_mvp.fato_contexto_pais_ano c
       ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
  GROUP BY g.country, t.year, c.fuel_price_usd_per_liter, c.electricity_price_usd_per_kwh
);

Q5 - Renda e urbanização vs. adoção

In [0]:
%sql
SELECT g.country, t.year,
       AVG(v.ev_market_share) AS share_ev,
       c.gdp_per_capita, c.urban_population_percent
FROM ev_mvp.fato_vendas v
JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
JOIN ev_mvp.fato_contexto_pais_ano c
     ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
GROUP BY g.country, t.year, c.gdp_per_capita, c.urban_population_percent
ORDER BY share_ev DESC;

In [0]:
%sql
SELECT
  ROUND(corr(share_ev, gdp_per_capita),           3) AS corr_renda,
  ROUND(corr(share_ev, urban_population_percent),  3) AS corr_urbanizacao
FROM (
  SELECT g.country, t.year,
         AVG(v.ev_market_share) AS share_ev,
         c.gdp_per_capita,
         c.urban_population_percent
  FROM ev_mvp.fato_vendas v
  JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
  JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
  JOIN ev_mvp.fato_contexto_pais_ano c
       ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
  GROUP BY g.country, t.year, c.gdp_per_capita, c.urban_population_percent
);

Q6 - Crescimento de EV vs. emissões de transporte

In [0]:
%sql
SELECT g.country, t.year,
       AVG(v.ev_market_share) AS share_ev,
       c.co2_emissions_transport_mt
FROM ev_mvp.fato_vendas v
JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
JOIN ev_mvp.fato_contexto_pais_ano c
     ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
GROUP BY g.country, t.year, c.co2_emissions_transport_mt
ORDER BY g.country, t.year;

In [0]:
%sql
SELECT ROUND(corr(share_ev, co2_emissions_transport_mt), 3) AS corr_global
FROM (
  SELECT g.country, t.year,
         AVG(v.ev_market_share) AS share_ev,
         c.co2_emissions_transport_mt
  FROM ev_mvp.fato_vendas v
  JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
  JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
  JOIN ev_mvp.fato_contexto_pais_ano c
       ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
  GROUP BY g.country, t.year, c.co2_emissions_transport_mt
);

In [0]:
%sql
SELECT country,
       ROUND(corr(year, share_ev), 2)                       AS corr_ano_share,
       ROUND(corr(share_ev, co2_emissions_transport_mt), 2) AS corr_share_co2
FROM (
  SELECT g.country, t.year,
         AVG(v.ev_market_share) AS share_ev,
         c.co2_emissions_transport_mt
  FROM ev_mvp.fato_vendas v
  JOIN ev_mvp.dim_geografia g ON v.sk_geografia = g.sk_geografia
  JOIN ev_mvp.dim_tempo     t ON v.sk_tempo     = t.sk_tempo
  JOIN ev_mvp.fato_contexto_pais_ano c
       ON v.sk_geografia = c.sk_geografia AND v.sk_tempo = c.sk_tempo
  GROUP BY g.country, t.year, c.co2_emissions_transport_mt
)
GROUP BY country
ORDER BY country;

Q7 - Segmento que lidera a transição

In [0]:
%sql
SELECT s.vehicle_segment, ROUND(AVG(f.ev_market_share), 2) AS share_medio
FROM ev_mvp.fato_vendas f
JOIN ev_mvp.dim_segmento s ON f.sk_segmento = s.sk_segmento
GROUP BY s.vehicle_segment
ORDER BY share_medio DESC;